# Part 1 — Data fetch & parquet build (survivorship-corrected, through 2026)

Drop-in replacement for `01_Data_Prep.ipynb`. Output schema is **compatible with the training notebook**, plus one new column (`InIndex`).

### What changed and why

**1. Survivorship bias — the important one.** The old notebook scraped *today's* S&P 500 list and pulled each name's history back to 1997. That makes the 2005 "universe" equal to *the firms still in the index in 2026* — Lehman, Bear Stearns, Countrywide, Sears, Albertsons, GM and hundreds of others simply vanish. They skew to the losing side, so their absence **inflates the long-short return and Sharpe**. JKX avoid this by using a point-in-time CRSP universe.

This notebook instead reconstructs **point-in-time membership** from a maintained daily snapshot file (1996 → 2026, ~1,200 tickers ever vs. ~500 today) and tags each row with `InIndex` — was this stock actually in the index on that date. See §6 for how honest the fix really is.

**2. Dates.** The old default already ran to `2026-05-31` (despite the `1997_2023` filename), but the *training* notebook then threw away everything after 2023 with a hardcoded test window. Both ends are now explicit and consistent.

**3. No blanket `dropna()`.** The old final line dropped every row with any NaN, which deletes each ticker's first ~252 days (MOM's warm-up) — so the CNN could never see early history, even though the paper's CNN reads **only the image**. Now: rows are dropped only when OHLCV is unusable. Signals stay NaN during warm-up; the training notebook's scorer already drops NaN signals per benchmark.

## 1. Imports

In [18]:
import os, time, io, requests
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

# ---------------- knobs ----------------
START_DATE   = '1997-01-01'
END_DATE     = '2026-07-01'          # exclusive upper bound for yfinance
CHUNK_SIZE   = 20
SLEEP_SEC    = 1.0                   # be polite to Yahoo

POINT_IN_TIME = True                 # True  -> reconstruct historical membership (fixes survivorship bias)
                                     # False -> old behaviour: today's S&P 500 list, back-projected
MIN_HISTORY   = 252                  # skip tickers with under a year of data

OUT_FILE = 'market_data_sp500_1997_2026.parquet'
print(f"{START_DATE} -> {END_DATE} | point_in_time={POINT_IN_TIME} | out={OUT_FILE}")
# --- Cell C prerequisites (safe to re-declare) ---
IMAGE_SIZES   = globals().get('IMAGE_SIZES',   [5, 20, 60])
HORIZONS      = globals().get('HORIZONS',      [5, 20, 60])
ENSEMBLE_SIZE = globals().get('ENSEMBLE_SIZE', 5)

1997-01-01 -> 2026-07-01 | point_in_time=True | out=market_data_sp500_1997_2026.parquet


## 2. Universe — point-in-time membership

`fja05680/sp500` maintains a snapshot of index membership on every change date since 1996. We turn that into **membership intervals** per ticker: `(ticker, start, end)`. A stock is only a valid *sample* on dates it was actually a member — but we still download its full price history, because a window ending on a member date needs the prior `n_days` of prices regardless of whether it was in the index back then.

In [19]:
MEMBERSHIP_URL = ('https://raw.githubusercontent.com/fja05680/sp500/master/'
                  'S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv')

def get_current_tickers():
    """Fallback / POINT_IN_TIME=False path: today's list from Wikipedia."""
    hdr = {'User-Agent': 'Mozilla/5.0'}
    html = requests.get('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', headers=hdr).text
    syms = pd.read_html(io.StringIO(html))[0]['Symbol'].tolist()
    return sorted({s.replace('.', '-') for s in syms})

def get_membership_intervals():
    """-> DataFrame[ticker, start, end]: the dates each ticker was an S&P 500 member."""
    snaps = pd.read_csv(MEMBERSHIP_URL)
    snaps['date'] = pd.to_datetime(snaps['date'])
    snaps = snaps.sort_values('date').reset_index(drop=True)
    print(f"membership snapshots: {len(snaps):,} change dates "
          f"({snaps['date'].min().date()} -> {snaps['date'].max().date()})")

    members = [(d, set(t.strip() for t in s.split(',') if t.strip()))
               for d, s in zip(snaps['date'], snaps['tickers'])]
    open_at, rows = {}, []
    for i, (d, cur) in enumerate(members):
        prev = members[i-1][1] if i else set()
        for t in cur - prev:                       # joined on d
            open_at[t] = d
        for t in prev - cur:                       # left on d
            if t in open_at:
                rows.append({'ticker': t, 'start': open_at.pop(t), 'end': d})
    last = members[-1][0]
    for t, s in open_at.items():                   # still a member at the end of the file
        rows.append({'ticker': t, 'start': s, 'end': pd.Timestamp('2100-01-01')})

    iv = pd.DataFrame(rows)
    iv['ticker'] = iv['ticker'].str.replace('.', '-', regex=False)   # yfinance uses BRK-B, not BRK.B
    iv = iv[iv['end'] >= pd.Timestamp(START_DATE)]                   # irrelevant to our sample window
    print(f"membership intervals: {len(iv):,} | unique tickers ever: {iv['ticker'].nunique():,}")
    return iv

if POINT_IN_TIME:
    intervals = get_membership_intervals()
    tickers   = sorted(intervals['ticker'].unique())
    still_in  = intervals[intervals['end'] > pd.Timestamp('2026-01-01')]['ticker'].nunique()
    print(f"\nUNIVERSE: {len(tickers):,} tickers ever in the index during our window")
    print(f"  of which still members in 2026: {still_in:,}")
    print(f"  -> {len(tickers)-still_in:,} names the old notebook silently dropped (this is the bias)")
else:
    intervals = None
    tickers   = get_current_tickers()
    print(f"UNIVERSE (survivorship-biased): {len(tickers):,} current members")

membership snapshots: 2,712 change dates (1996-01-02 -> 2026-06-02)
membership intervals: 1,237 | unique tickers ever: 1,185

UNIVERSE: 1,185 tickers ever in the index during our window
  of which still members in 2026: 513
  -> 672 names the old notebook silently dropped (this is the bias)


## 3. Download

Delisted tickers are the whole point of the exercise, and they are exactly the ones Yahoo is worst at. Failures are counted, not hidden — the success rate in §6 is your honest measure of how much bias remains.

In [20]:
def download(tickers, start, end, chunk=CHUNK_SIZE):
    frames, missing = [], []
    for i in tqdm(range(0, len(tickers), chunk), desc="downloading"):
        grp = tickers[i:i+chunk]
        try:
            raw = yf.download(grp, start=start, end=end, progress=False,
                              group_by='ticker', auto_adjust=False, threads=True)
        except Exception as e:
            print(f"  chunk {i} failed: {e}"); missing += grp; continue
        time.sleep(SLEEP_SEC)

        for t in grp:
            try:
                d = raw.copy() if len(grp) == 1 else (
                    raw[t].copy() if t in raw.columns.get_level_values(0) else None)
            except Exception:
                d = None
            if d is None or d.empty:
                missing.append(t); continue
            if 'Adj Close' not in d.columns:          # newer yfinance may auto-adjust and drop it
                d['Adj Close'] = d['Close']
            d = d.dropna(subset=['Open','High','Low','Close','Adj Close','Volume'])
            if len(d) < MIN_HISTORY:
                missing.append(t); continue
            d['Ticker'] = t
            frames.append(d.reset_index()[['Date','Ticker','Open','High','Low','Close','Adj Close','Volume']])
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), missing

raw_df, missing = download(tickers, START_DATE, END_DATE)
print(f"\ndownloaded {len(raw_df):,} rows for {raw_df['Ticker'].nunique():,} tickers")
print(f"unavailable: {len(missing):,} tickers (mostly delisted names Yahoo no longer serves)")

downloading:   0%|          | 0/60 [00:00<?, ?it/s]

$ACAS: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
$ABKFQ: possibly delisted; no timezone found
$ABS: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
$ABMD: possibly delisted; no timezone found
$AAMRQ: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
$ACS: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
$ACKH: possibly delisted; no timezone found
$AABA: possibly delisted; no timezone found
$ABC: possibly delisted; no timezone found

9 Failed downloads:
['ACAS', 'ABS', 'AAMRQ', 'ACS']: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
['ABKFQ', 'ABMD', 'ACKH', 'AABA', 'ABC']: possibly delisted; no timezone found
$AHM: possibly delisted; no timezone found
$AFS-A: possibly delisted; no timezone found
$AGC: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$ADS: possibly delisted; no timezone found

5 Failed downloads:
['AHM', 'AFS-A', 'AGC', 'AGN', '


downloaded 4,329,318 rows for 742 tickers
unavailable: 443 tickers (mostly delisted names Yahoo no longer serves)


In [21]:
# =============================================================================
# SURVIVORSHIP ADD-ONS
#
# CELL A + B  -> paste into 01_Data_Prep_v2.ipynb  (A after §3 download, B after §6)
# CELL C      -> paste into notebook_final_unified.ipynb (after the main run)
#                It RE-SCORES the cached predictions. No retraining. Costs seconds.
# =============================================================================


# ============================== CELL A: retry failed tickers =================
# Paste AFTER the §3 download cell. Then RE-RUN §4, §5, §6 so the new rows flow through.

def symbol_variants(t):
    """Yahoo's symbology for old / odd tickers differs from the index files."""
    v = {t}
    v.add(t.replace('-', '.'))            # BRK-B -> BRK.B
    v.add(t.replace('.', '-'))            # BF.B  -> BF-B
    if len(t) > 1 and t.endswith('Q'):    # bankruptcy suffix: AAMRQ -> AAMR
        v.add(t[:-1])
    for suf in ('.A', '.B', '-A', '-B'):
        if t.endswith(suf):
            v.add(t[:-2])
    return [x for x in v if x and x != t]

print(f"retrying {len(missing):,} failed tickers with symbol variants...")
recovered_frames, still_missing = [], []

for t in tqdm(missing, desc="retry"):
    got = None
    for cand in symbol_variants(t):
        try:
            d = yf.download(cand, start=START_DATE, end=END_DATE,
                            progress=False, auto_adjust=False)
        except Exception:
            continue
        if d is None or d.empty:
            continue
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)
        if 'Adj Close' not in d.columns:
            d['Adj Close'] = d['Close']
        d = d.dropna(subset=['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'])
        if len(d) < MIN_HISTORY:
            continue
        d['Ticker'] = t                   # KEEP the index-file symbol so membership joins still work
        got = d.reset_index()[['Date','Ticker','Open','High','Low','Close','Adj Close','Volume']]
        break
    time.sleep(0.3)
    if got is not None:
        recovered_frames.append(got)
    else:
        still_missing.append(t)

if recovered_frames:
    extra = pd.concat(recovered_frames, ignore_index=True)
    raw_df = pd.concat([raw_df, extra], ignore_index=True)
    print(f"\nrecovered {len(recovered_frames):,} tickers ({len(extra):,} rows) via symbol variants")
else:
    print("\nno additional tickers recovered")
print(f"still unavailable: {len(still_missing):,}")
missing = still_missing
print("\n>>> NOW RE-RUN cells §4 (adjust_ohlc), §5 (signals), §6 (InIndex) <<<")

retrying 443 failed tickers with symbol variants...


retry:   0%|          | 0/443 [00:00<?, ?it/s]

$AAMR: possibly delisted; no timezone found

1 Failed download:
['AAMR']: possibly delisted; no timezone found
$ABKF: possibly delisted; no timezone found

1 Failed download:
['ABKF']: possibly delisted; no timezone found
$AFS: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)

1 Failed download:
['AFS']: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
$AFS.A: possibly delisted; no timezone found

1 Failed download:
['AFS.A']: possibly delisted; no timezone found
$ANRZ: possibly delisted; no timezone found

1 Failed download:
['ANRZ']: possibly delisted; no timezone found
$AZA.A: possibly delisted; no timezone found

1 Failed download:
['AZA.A']: possibly delisted; no timezone found
$AZA: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)

1 Failed download:
['AZA']: possibly delisted; no price data found  (1d 1997-01-01 -> 2026-07-01)
$BHMS: possibly delisted; no timezone found

1 Failed download:
['BHMS']: possibly delisted


recovered 6 tickers (21,566 rows) via symbol variants
still unavailable: 437

>>> NOW RE-RUN cells §4 (adjust_ohlc), §5 (signals), §6 (InIndex) <<<


## 4. Adjust OHLC

The paper abstracts from splits and dividends, so scale O/H/L by `Adj Close / Close` and use `Adj Close` as the close. (With `auto_adjust=False` above, `Adj Close` really is present; if a future yfinance drops it we fall back to `Close`, making the ratio 1 — a safe no-op, since in that case the prices are *already* adjusted.)

In [26]:
def adjust_ohlc(df):
    df = df.copy()
    ratio = (df['Adj Close'] / df['Close']).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    for c in ['Open','High','Low']:
        df[c] = df[c] * ratio
    df['Close'] = df['Adj Close']
    return df.drop(columns=['Adj Close'])

clean_df = adjust_ohlc(raw_df)
clean_df = clean_df[clean_df['Close'] > 0].sort_values(['Ticker','Date']).reset_index(drop=True)
print(f"adjusted: {len(clean_df):,} rows | {clean_df['Date'].min().date()} -> {clean_df['Date'].max().date()}")

adjusted: 4,350,152 rows | 1997-01-02 -> 2026-06-30


## 5. Benchmark signals

**`groupby(...).apply()` is gone.** In pandas 2.2+/3.0 the grouping column is *excluded* from the frame handed to the function, so `Ticker` disappeared and `reset_index(drop=True)` then discarded the key — that was the `KeyError: 'Ticker'`. Everything below is vectorized `groupby` instead: correct on every pandas version, and far faster.

**Signal definitions** — each is oriented so that **higher = go long**, which is what makes a "long the top decile, short the bottom" sort meaningful and comparable across signals:

| Signal | Definition | Source |
|---|---|---|
| `MOM` | return from t−252 to t−21 (2–12 month momentum, skipping the last month) | Jegadeesh & Titman (1993) |
| `STR` | **−**(return over the last 21 days) | Jegadeesh (1990) reversal |
| `WSTR` | **−**(return over the last 5 days) | Lehmann (1990) reversal |
| `MA{L}` | MA of close over L days ÷ current close, for L ∈ {3,5,10,20,50,100,200} | inputs to the trend factor |

MOM/STR/WSTR match the paper. **TREND is no longer computed here.** The old code used a moving-average *crossover vote* while the markdown credited Han, Zhou & Zhu (2016) — but HZZ's trend factor is a cross-sectionally **fitted expected-return forecast**, not a vote. The genuine factor needs (a) the forward-return horizon and (b) betas estimated only from *already-realized* returns, so it is computed in the training notebook where the horizon is known. Here we just emit its raw inputs, the normalized moving averages.

In [27]:
MA_LAGS = [3, 5, 10, 20, 50, 100, 200]
MA_COLS = [f'MA{L}' for L in MA_LAGS]

final_df = clean_df.sort_values(['Ticker','Date']).reset_index(drop=True)
grp = final_df.groupby('Ticker')['Close']

# --- momentum & reversals (oriented so higher = long) ---
final_df['MOM']  = grp.shift(21) / grp.shift(252) - 1        # 2-12 month momentum
final_df['STR']  = -(final_df['Close'] / grp.shift(21)  - 1) # 1-month reversal
final_df['WSTR'] = -(final_df['Close'] / grp.shift(5)   - 1) # 1-week reversal

# --- normalized moving averages: the raw inputs to the Han-Zhou-Zhu trend factor ---
for L in MA_LAGS:
    ma = final_df.groupby('Ticker')['Close'].transform(lambda s, L=L: s.rolling(L).mean())
    final_df[f'MA{L}'] = ma / final_df['Close']

assert 'Ticker' in final_df.columns, "Ticker column lost -- groupby regression"

# Drop ONLY rows whose image cannot be built. Signal warm-up NaNs are KEPT on purpose:
# the CNN reads the image alone, and the scorer drops NaN signals per benchmark.
before = len(final_df)
final_df = final_df.dropna(subset=['Open','High','Low','Close','Volume']).reset_index(drop=True)
print(f"rows: {before:,} -> {len(final_df):,} (dropped {before-len(final_df):,} unusable OHLCV rows)")
print("signal NaNs retained (warm-up):",
      {c: int(final_df[c].isna().sum()) for c in ['MOM','STR','WSTR'] + MA_COLS[:2]})
display(final_df[['Ticker','Date','Close','MOM','STR','WSTR'] + MA_COLS].tail(3))

rows: 4,350,152 -> 4,350,152 (dropped 0 unusable OHLCV rows)
signal NaNs retained (warm-up): {'MOM': 188496, 'STR': 15708, 'WSTR': 3740, 'MA3': 1496, 'MA5': 2992}


Price,Ticker,Date,Close,MOM,STR,WSTR,MA3,MA5,MA10,MA20,MA50,MA100,MA200
4350149,ZTS,2026-06-26,76.089996,-0.486646,0.042652,0.033287,1.016691,1.011460,1.024616,1.032678,1.182990,1.390223,1.551724
4350150,ZTS,2026-06-29,73.250000,-0.489788,0.064137,0.034787,1.033720,1.043468,1.055713,1.069686,1.216494,1.437485,1.606809
4350151,ZTS,2026-06-30,71.860001,-0.494801,0.075042,0.064810,1.026069,1.049791,1.064876,1.086411,1.226112,1.457736,1.632536


## 6. Tag point-in-time membership + honesty report

`InIndex` = was this stock an S&P 500 member on this date. Price history *outside* membership is kept so a window ending on a member date still has its full lookback — but only `InIndex` rows should become training/eval samples (see §8 for the one-line patch).

In [28]:
if POINT_IN_TIME:
    final_df['InIndex'] = False
    iv_by_t = {t: g[['start','end']].values for t, g in intervals.groupby('ticker')}
    for t, pos in tqdm(final_df.groupby('Ticker').indices.items(), desc="tagging membership"):
        spans = iv_by_t.get(t)
        if spans is None: continue
        d = final_df['Date'].values[pos]
        flag = np.zeros(len(pos), bool)
        for s, e in spans:
            flag |= (d >= np.datetime64(s)) & (d < np.datetime64(e))
        final_df.loc[final_df.index[pos], 'InIndex'] = flag
else:
    final_df['InIndex'] = True        # every row treated as a member (the biased assumption)

n_mem = int(final_df['InIndex'].sum())
print(f"\nrows in-index: {n_mem:,} / {len(final_df):,} ({100*n_mem/len(final_df):.1f}%)")

print("\n" + "="*64)
print("SURVIVORSHIP HONESTY REPORT — how much bias actually remains")
print("="*64)
if POINT_IN_TIME:
    ever   = set(tickers)
    got    = set(final_df['Ticker'].unique())
    lost   = ever - got
    # names that LEFT the index before the end and so are pure survivorship risk
    dead   = set(intervals[intervals['end'] < pd.Timestamp('2026-01-01')]['ticker'])
    dead_got, dead_lost = dead & got, dead - got
    print(f"tickers ever in index      : {len(ever):,}")
    print(f"  downloaded OK            : {len(got):,}  ({100*len(got)/len(ever):.0f}%)")
    print(f"  unavailable on Yahoo     : {len(lost):,}")
    print(f"de-indexed (delisted etc.) : {len(dead):,}")
    print(f"  ...recovered             : {len(dead_got):,}  <- these fix the bias")
    print(f"  ...still missing         : {len(dead_lost):,}  <- RESIDUAL bias")
    print(f"\nYahoo does not serve most fully-delisted symbols, so the fix is PARTIAL.")
    print(f"Report this: 'point-in-time membership recovered {len(dead_got):,} of {len(dead):,} "
          f"de-indexed names; {len(dead_lost):,} remain unavailable.'")
    # cross-sectional breadth over time -- a healthy panel sits near ~500
    yr = final_df[final_df['InIndex']].groupby(final_df['Date'].dt.year)['Ticker'].nunique()
    print("\nin-index names per year (should hover near 500):")
    print(yr.to_string())
else:
    print("POINT_IN_TIME=False -> universe is today's members, back-projected.")
    print("Long-short returns and Sharpe are BIASED UPWARD. Do not present as a clean replication.")

tagging membership:   0%|          | 0/748 [00:00<?, ?it/s]


rows in-index: 2,587,847 / 4,350,152 (59.5%)

SURVIVORSHIP HONESTY REPORT — how much bias actually remains
tickers ever in index      : 1,185
  downloaded OK            : 748  (63%)
  unavailable on Yahoo     : 437
de-indexed (delisted etc.) : 703
  ...recovered             : 274  <- these fix the bias
  ...still missing         : 429  <- RESIDUAL bias

Yahoo does not serve most fully-delisted symbols, so the fix is PARTIAL.
Report this: 'point-in-time membership recovered 274 of 703 de-indexed names; 429 remain unavailable.'

in-index names per year (should hover near 500):
Date
1997    213
1998    227
1999    238
2000    251
2001    255
2002    264
2003    268
2004    272
2005    282
2006    295
2007    313
2008    332
2009    338
2010    341
2011    350
2012    360
2013    370
2014    374
2015    393
2016    414
2017    429
2018    436
2019    449
2020    457
2021    467
2022    478
2023    491
2024    500
2025    507
2026    504


In [29]:
# ============================== CELL B: coverage by year ====================
# Paste AFTER §6. This is the honest denominator: how much of the TRUE index we actually have.

def coverage_report(final_df, intervals, oos_start=2005):
    got = set(final_df['Ticker'].unique())
    rows = []
    for y in sorted(final_df['Date'].dt.year.unique()):
        d = pd.Timestamp(f'{y}-07-01')                       # mid-year snapshot
        live   = intervals[(intervals['start'] <= d) & (intervals['end'] > d)]['ticker']
        true_n = live.nunique()
        have_n = live[live.isin(got)].nunique()
        rows.append({'year': y, 'true_members': true_n, 'available': have_n,
                     'coverage_%': 100.0 * have_n / max(true_n, 1)})
    cov = pd.DataFrame(rows)

    print("=" * 60)
    print("COVERAGE BY YEAR — available in-index names vs true membership")
    print("=" * 60)
    print(cov.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

    oos = cov[cov['year'] >= oos_start]
    drift = oos['coverage_%'].iloc[-1] - oos['coverage_%'].iloc[0]
    print(f"\nOOS window ({oos_start}+):")
    print(f"  mean coverage : {oos['coverage_%'].mean():.1f}%")
    print(f"  range         : {oos['coverage_%'].min():.0f}% -> {oos['coverage_%'].max():.0f}%")
    print(f"  min names/yr  : {int(oos['available'].min())} "
          f"(>=100 gives usable deciles; >=200 is comfortable)")
    print(f"  DRIFT         : {drift:+.1f} pp across the OOS window")
    print("\n  The drift is the real issue: late years are near-complete while early years")
    print("  are survivor-selected, so performance is NOT measured on a constant universe.")
    print("\n  Note the direction of the bias is genuinely ambiguous for a LONG-SHORT book:")
    print("  most index departures are acquisitions (winners, takeover premium), not")
    print("  bankruptcies (losers). So the honest claim is 'the universe is incomplete and")
    print("  time-varying, so the LEVEL of the Sharpe is not trustworthy' -- NOT 'the Sharpe")
    print("  is inflated'. The sub-period test (Cell C) is what settles whether it matters.")
    return cov

cov = coverage_report(final_df, intervals)

COVERAGE BY YEAR — available in-index names vs true membership
 year  true_members  available  coverage_%
 1997           488        283        58.0
 1998           490        290        59.2
 1999           491        291        59.3
 2000           492        294        59.8
 2001           495        304        61.4
 2002           495        308        62.2
 2003           494        316        64.0
 2004           495        319        64.4
 2005           496        319        64.3
 2006           497        330        66.4
 2007           497        335        67.4
 2008           497        343        69.0
 2009           499        352        70.5
 2010           499        359        71.9
 2011           497        364        73.2
 2012           497        368        74.0
 2013           497        378        76.1
 2014           498        382        76.7
 2015           500        391        78.2
 2016           504        411        81.5
 2017           506        422    

## 7. Save

In [30]:
cols = (['Date','Ticker','Open','High','Low','Close','Volume','MOM','STR','WSTR']
        + MA_COLS + ['InIndex'])
out = final_df[cols].sort_values(['Ticker','Date']).reset_index(drop=True)
out['Date'] = pd.to_datetime(out['Date'])
out.to_parquet(OUT_FILE, index=False)

print(f"saved {OUT_FILE}: {len(out):,} rows | {out['Ticker'].nunique():,} tickers "
      f"| {out['Date'].min().date()} -> {out['Date'].max().date()} "
      f"| {os.path.getsize(OUT_FILE)/1e6:.0f} MB")
display(out.head())

saved market_data_sp500_1997_2026.parquet: 4,350,152 rows | 748 tickers | 1997-01-02 -> 2026-06-30 | 511 MB


Price,Date,Ticker,Open,High,Low,Close,Volume,MOM,STR,WSTR,MA3,MA5,MA10,MA20,MA50,MA100,MA200,InIndex
0,1999-11-18,A,27.082606,29.761108,23.808885,26.189774,62546380.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1999-11-19,A,25.557348,25.594550,23.697280,24.032093,15234146.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1999-11-22,A,24.590113,26.189774,23.846086,26.189774,6577870.0,NaN,NaN,NaN,0.972538,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1999-11-23,A,25.296945,25.966571,23.808889,23.808889,5975611.0,NaN,NaN,NaN,1.036458,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1999-11-24,A,23.883290,24.962131,23.808886,24.441311,4843231.0,NaN,NaN,NaN,1.015221,1.020091,NaN,NaN,NaN,NaN,NaN,False


## 8. Wiring it into the training notebook

Use **`notebook_final_unified_v2.ipynb`** — it already carries every matching change:

- `PARQUET_PATH` -> `market_data_sp500_1997_2026.parquet`
- OOS window extended to **2026-12-31**
- samples restricted to `InIndex` dates (lookback prices from non-member days are still used -- they are real prices; only the *sample* must be an index member)
- the `MA{L}` columns are carried through the image store, and **TREND is computed at scoring time** as the true Han-Zhou-Zhu factor for the given horizon

This changes the images, so delete any existing `raw_n*.dat` / `meta_n*.parquet` and regenerate.